[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C58_HardCase_LongTail_Course/04_mining_infra/04_mining_infrastructure.ipynb)

# 04 · 大规模挖掘基础设施（嵌入检索 / ANN 索引 / 场景打标 / 去重与多样性 / 预算分配 / 血缘）

目标：把「车队每天回传几十 TB，标注预算只够几千张」这个真问题，拆成五个可以写代码解决的子问题，
并亲手实现每一个。

路线：合成候选池 → **嵌入检索**（找和 badcase 像的）→ IVF 索引的召回-代价旋钮 →
**感知哈希去重** → **core-set 贪心多样性采样** → 元数据/规则/**VLM 级联打标** →
**标注预算分配** → ✏️ 练习（MMR / 分层配额 / 近重复分组 / 数据血缘）→ 📖 答案 → 🧪 工程胶囊。

本 notebook 你会亲手实现：
- 余弦 kNN 检索 + `precision@k` 评估，并量化「不做 L2 归一化」的代价
- IVF 倒排索引（k-means 粗量化 + nprobe），画出**召回 vs 距离计算次数**的旋钮
- aHash 感知哈希 + 贪心去重，量化 clip 内 / clip 间的汉明距离分布
- k-center 贪心 core-set 选择，对比随机采样与 top-k 检索的**覆盖度**
- 元数据规则 + 模拟 VLM 打标 + **级联**，算清成本-精度账（含逐取值召回与 Cohen's kappa）
- 带成本的贪心**标注预算分配**，与平均分 / 缺口最大优先做对比

> 心智模型：**相似度决定「找什么」，多样性决定「买什么」。
> 把这两个阶段的目标搞混，就是「回传 1000 张同一路口」的根本原因。**

## 1 · 合成一个车队候选池

规则：每个 <strong>clip</strong>（一段连续行车）有固定的（天气, 光照, 道路, 标志类别），
clip 内 3–8 帧几乎相同（这就是近重复的来源）。嵌入 = 域中心 + 类别中心 + 噪声。
另外**注入 10 段「雾天隧道口施工牌」**——这是我们这一轮要挖的失效场景。

In [ ]:
import numpy as np, itertools, collections, math, json
rng = np.random.default_rng(58)

WEATHER = ['clear', 'rain', 'fog', 'snow']
LIGHT   = ['day', 'dusk', 'night', 'tunnel']
ROAD    = ['highway', 'urban', 'ramp']
SIGN    = ['speed_60', 'speed_80', 'stop', 'no_left', 'construction']
D = 32                                    # 嵌入维度（真实系统 256~2048）

dom_center  = {k: rng.normal(size=D) for k in itertools.product(WEATHER, LIGHT)}
sign_center = {s: rng.normal(size=D) for s in SIGN}

def l2norm(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

frames = []
def add_clip(cid, w, l, r, s):
    base = 0.9 * dom_center[(w, l)] + 1.0 * sign_center[s] + 0.35 * rng.normal(size=D)
    bg = rng.normal(size=(4, 4)) * 0.22          # 每个 clip 独有的背景（渲染时用）
    for _ in range(int(rng.integers(3, 9))):     # clip 内 3~8 帧 = 近重复
        frames.append(dict(clip=cid, weather=w, light=l, road=r, sign=s, bg=bg,
                           emb=base + 0.06 * rng.normal(size=D),
                           jit=rng.normal(size=2) * 0.6))

N_CLIPS = 380
for c in range(N_CLIPS):
    add_clip(c, WEATHER[rng.choice(4, p=[.70, .15, .09, .06])],
                LIGHT[rng.choice(4,   p=[.58, .16, .21, .05])],
                ROAD[rng.choice(3,    p=[.35, .55, .10])],
                SIGN[rng.choice(5,    p=[.44, .26, .15, .10, .05])])
TARGET = ('fog', 'tunnel', 'construction')       # ← 本轮要挖的失效场景
for c in range(N_CLIPS, N_CLIPS + 10):
    add_clip(c, TARGET[0], TARGET[1], 'highway', TARGET[2])

E = l2norm(np.array([f['emb'] for f in frames]))  # **先 L2 归一化**：点积 = 余弦
N = len(frames)
clip_of = np.array([f['clip'] for f in frames])
print(f'候选池: {N} 帧 / {N_CLIPS + 10} 个 clip，嵌入维度 {D}')

cnt = collections.Counter((f['weather'], f['light']) for f in frames)
print('\n最常见 4 个 (天气,光照) 组合:')
for k, v in cnt.most_common(4):
    print(f'  {str(k):<22s} {v:5d} 帧  {v / N:6.1%}')
tail = set(k for k, v in cnt.items() if v / N < 0.02)
print(f'\n占比 <2% 的长尾组合: {len(tail)} 个，合计只占 {sum(cnt[k] for k in tail) / N:.1%}')
assert N > 1500 and len(tail) >= 3
print('✅ 这就是长尾：头部两三个组合占掉一半以上，几十个尾部组合分剩下的零头')

## 2 · 嵌入检索：找「和这个 badcase 长得像的」

评估必须**排除查询帧自己所在的 clip**——否则等于自问自答（同 clip 的帧几乎是同一张图）。
`precision@k` 要和「随机抽 k 张」的基线比，**看的是倍数，不是绝对值**。

In [ ]:
def knn(q, X, k=10, banned=None):
    '''余弦 kNN。X 已 L2 归一化 -> 点积即余弦相似度。'''
    sims = X @ q
    if banned is not None:
        sims = sims.copy(); sims[banned] = -np.inf
    k = min(k, int(np.isfinite(sims).sum()))
    idx = np.argpartition(-sims, k - 1)[:k]
    return idx[np.argsort(-sims[idx])], sims

def cell(f):                       # 场景「格子」= (天气, 光照, 标志类别)
    return (f['weather'], f['light'], f['sign'])

qi = int(np.where(clip_of == N_CLIPS)[0][0])          # 一个 badcase：雾天隧道口的施工牌
banned = clip_of == frames[qi]['clip']                # **排除自己那一段**
rel = np.array([cell(f) == cell(frames[qi]) for f in frames]) & (~banned)
base_rate = float(rel.mean())
order, _ = knn(E[qi], E, k=30, banned=banned)
hit = rel[order]

print('badcase 查询帧:', {k: frames[qi][k] for k in ['clip', 'weather', 'light', 'road', 'sign']})
print(f'池中同格子且不同 clip 的帧: {int(rel.sum())} / {N}  -> 随机基线 {base_rate:.2%}\n')
for k in [5, 10, 20, 30]:
    print(f'  precision@{k:<3d} = {hit[:k].mean():6.1%}   （随机基线 {base_rate:.2%}，'
          f'提升 {hit[:k].mean() / base_rate:5.1f}×）')
n_clip_top = len(set(clip_of[order].tolist()))
print(f'\n⚠️  top-30 只来自 {n_clip_top} 个不同的 clip —— 检索**天然会扎堆**，')
print('    直接把 top-k 送标就是「1000 张同一路口」。这正是第 4、5 节要解决的问题。')
assert hit[:10].mean() > 10 * base_rate, '嵌入检索应把命中率提升一个数量级以上'
assert n_clip_top < 15
print('✅ 以图搜图把命中率提升了一个数量级 —— 这是长尾挖掘的主力工具')
print('⚠️  合成数据是干净可分的，真实系统的 precision@10 通常只有 0.3~0.6；')
print('    **要看的是「比随机基线高多少倍」，而不是绝对值**。低于 3× 就别建索引了。')

In [ ]:
# ⚠️ 不做 L2 归一化的代价：裸内积会被「向量模长大」的帧劫持
scale = np.exp(rng.normal(0, 0.7, size=N))[:, None]   # 不同曝光/不同层输出 -> 模长差异
E_raw = E * scale

qs = rng.choice(N, 200, replace=False)
p_cos, p_dot = [], []
for q_ in qs:
    b_ = clip_of == frames[q_]['clip']
    r_ = np.array([cell(f) == cell(frames[q_]) for f in frames]) & (~b_)
    if r_.sum() < 5:
        continue
    o1, _ = knn(E[q_], E, k=10, banned=b_)
    sraw = E_raw @ E_raw[q_]; sraw[b_] = -np.inf
    o2 = np.argsort(-sraw)[:10]
    p_cos.append(r_[o1].mean()); p_dot.append(r_[o2].mean())

print(f'查询数 {len(p_cos)}')
print(f'  余弦（先 L2 归一化）  precision@10 = {np.mean(p_cos):.2%}')
print(f'  裸内积（不归一化）    precision@10 = {np.mean(p_dot):.2%}')
assert np.mean(p_cos) > 1.5 * np.mean(p_dot)
print('\n⚠️  裸内积的排序 = 模长 × 余弦。查询帧的模长对所有候选是常数，')
print('    所以排序被**候选的模长**主导 —— 高亮度/高响应的帧无脑排前面。')
print('✅ 建索引前一定先 L2 归一化（或用真正的 L2 距离），这是零成本的一行代码。')

## 3 · 向量索引：IVF 的「召回 vs 代价」旋钮

`nprobe` 是这个数据结构最重要的参数：它把「扫多少数据」变成一个**连续可调的旋钮**。
挖掘任务通常可以接受 0.9 的召回换 10× 的速度——**但这个取舍必须被量化，而不是拍脑袋**。

In [ ]:
def kmeans_cos(X, k, iters=20, seed=0):
    '''球面 k-means：粗量化器。'''
    r = np.random.default_rng(seed)
    C = X[r.choice(len(X), k, replace=False)].copy()
    for _ in range(iters):
        a = np.argmax(X @ C.T, axis=1)
        for j in range(k):
            m = (a == j)
            if m.any():
                C[j] = l2norm(X[m].mean(0))
    return C, np.argmax(X @ C.T, axis=1)

NLIST = 64                                   # 经验值 nlist ≈ sqrt(N)
C, assign = kmeans_cos(E, NLIST, seed=1)
lists = {j: np.where(assign == j)[0] for j in range(NLIST)}

def ivf_search(q, nprobe, k=10):
    cs = np.argsort(-(C @ q))[:nprobe]        # ① 先找最近的 nprobe 个粗中心
    cand = np.concatenate([lists[j] for j in cs])
    sims = E[cand] @ q                        # ② 只在这几个倒排列表里精算
    return cand[np.argsort(-sims)[:k]], len(cand) + NLIST   # 距离计算次数

QS = rng.choice(N, 80, replace=False)
gt = {i: set(knn(E[i], E, k=10)[0].tolist()) for i in QS}    # 暴力扫描的 ground truth
res = {}
print(f'{"nprobe":>7s} {"recall@10":>11s} {"距离计算次数":>13s} {"占暴力扫描":>11s}')
for npb in [1, 2, 4, 8, 16, NLIST]:
    rec = [len(set(ivf_search(E[i], npb)[0].tolist()) & gt[i]) / 10 for i in QS]
    cost = [ivf_search(E[i], npb)[1] for i in QS]
    res[npb] = (float(np.mean(rec)), float(np.mean(cost)))
    print(f'{npb:>7d} {res[npb][0]:>11.1%} {res[npb][1]:>13.0f} {res[npb][1] / N:>11.1%}')

assert res[1][0] < res[8][0], 'nprobe 越大召回越高'
assert res[NLIST][0] == 1.0, 'nprobe = nlist 时退化成暴力扫描，召回必然 1.0'
assert res[1][1] < 0.2 * N, 'nprobe=1 的代价应远小于暴力扫描'
print('\n✅ nprobe 给了一个**连续可调的召回-代价旋钮**：')
print(f'   nprobe=1 用 {res[1][1] / N:.0%} 的算力拿到 {res[1][0]:.0%} 的召回。')
print('⚠️  但注意：如果先用标签过滤再检索，命中的向量可能分散在很多桶里 ->')
print('    扫 nprobe 个桶一条也命中不到，**返回空结果且不报错**。')
print('    正确做法是把高频过滤维度**物化成分区**（按天气/光照分别建索引）。')

## 4 · 感知哈希：砍掉像素级近重复

先把每帧渲染成 32×32 的灰度「图像」（同 clip 内只差亚像素抖动），
再用 **aHash**（缩到 8×8，与均值比较取 0/1）得到 64 位指纹。

In [ ]:
def render(f, size=32):
    '''把一帧渲染成灰度图：标志盘面 + clip 特有背景 + 光照增益。'''
    yy, xx = np.mgrid[0:size, 0:size].astype(float)
    cx, cy = size / 2 + f['jit'][0], size / 2 + f['jit'][1]
    rad = {'stop': 7.0, 'speed_60': 10.0, 'speed_80': 10.0,
           'no_left': 8.5, 'construction': 9.2}[f['sign']]
    r = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    disc = r < rad
    img = np.where(disc, 0.95, 0.18)
    if   f['sign'] == 'speed_60':     img[disc & (xx < cx)] = 0.25
    elif f['sign'] == 'speed_80':     img[disc & (xx > cx)] = 0.25
    elif f['sign'] == 'no_left':      img[disc & (np.abs((xx - cx) + (yy - cy)) < 1.8)] = 0.25
    elif f['sign'] == 'construction': img[disc & (((xx - cx) * (yy - cy)) > 0)] = 0.35
    img = img + np.kron(f['bg'], np.ones((size // 4, size // 4)))
    gain = {'day': 1.0, 'dusk': 0.72, 'night': 0.38, 'tunnel': 0.55}[f['light']]
    return np.clip(img * gain, 0.0, 1.0)

def ahash(img):
    '''aHash：32x32 -> 8x8 块均值 -> 与整体均值比较 -> 64 bit。'''
    s = img.reshape(8, 4, 8, 4).mean(axis=(1, 3))
    return (s > s.mean()).ravel()

H = np.array([ahash(render(f)) for f in frames])
by_clip = collections.defaultdict(list)
for i, c in enumerate(clip_of):
    by_clip[c].append(i)

w_d = [int(np.count_nonzero(H[a] != H[b]))
       for idxs in by_clip.values() for a, b in itertools.combinations(idxs, 2)]
c_d = []
for _ in range(4000):
    i, j = rng.integers(0, N, 2)
    if clip_of[i] != clip_of[j]:
        c_d.append(int(np.count_nonzero(H[i] != H[j])))
print(f'clip **内**   汉明距离 均值 {np.mean(w_d):5.2f}  （{len(w_d)} 对）')
print(f'clip **之间** 汉明距离 均值 {np.mean(c_d):5.2f}  5% 分位 {np.percentile(c_d, 5):.0f}')
assert np.mean(w_d) < np.mean(c_d) / 4, '同 clip 的帧应显著更像'

def dedup_greedy(H, thresh):
    '''贪心：与已保留的任何一帧汉明距离 <= thresh 就丢弃。'''
    keep = []
    for i in range(len(H)):
        if all(np.count_nonzero(H[i] != H[j]) > thresh for j in keep):
            keep.append(i)
    return keep

print(f'\n{"阈值":>5s} {"保留帧数":>9s} {"压缩率":>8s} {"仍有代表的 clip":>16s}')
kept6 = None
for T in [0, 4, 6, 8]:
    k = dedup_greedy(H, T)
    if T == 6:
        kept6 = k
    print(f'{T:>5d} {len(k):>9d} {len(k) / N:>8.1%} {len(set(clip_of[k].tolist())):>10d} / {len(by_clip)}')
assert len(kept6) < N / 3 and len(set(clip_of[kept6].tolist())) > 0.8 * len(by_clip)
print('\n✅ 阈值 6 把候选池砍到 1/5 以下，而 85%+ 的 clip 仍有代表 ——')
print('   **丢掉的几乎全是「同一段路的相邻帧」，信息量损失极小、标注费省了 80%。**')
print('⚠️  阈值是要调的：太小压不动，太大会把「同一路口的雨天和晴天」也合并掉')
print('   —— 而那恰恰是你想要的对比样本。所以**跨场景标签不去重**。')

## 5 · core-set 贪心：把预算花在覆盖空白上

k-center 贪心：每次选「离已选集合最远的点」。目标是最小化**覆盖半径**
（池中任意点到最近被选点的最大距离）。它的行为和随机采样恰好相反——
随机按**密度**分配名额，k-center 按**空白**分配名额。

In [ ]:
def kcenter_greedy(X, k, start=0):
    sel = [start]
    d = 1.0 - X @ X[start]                     # 到已选集合的最小余弦距离
    radii = [float(d.max())]
    for _ in range(k - 1):
        i = int(np.argmax(d))                  # ← 覆盖得最差的那个点
        sel.append(i)
        d = np.minimum(d, 1.0 - X @ X[i])
        radii.append(float(d.max()))
    return np.array(sel), np.array(radii)

def coverage_radius(X, sel):
    return float(np.min(1.0 - X @ X[np.asarray(sel)].T, axis=1).max())

def bucket_cov(sel):
    return len({(frames[i]['weather'], frames[i]['light']) for i in sel})

def tail_frac(sel):
    return float(np.mean([(frames[i]['weather'], frames[i]['light']) in tail for i in sel]))

K = 80                                          # 本轮标注预算：80 张
sel_cs, radii = kcenter_greedy(E, K)
sel_topk, _ = knn(E[qi], E, k=K)                # 对照：直接送检索 top-K
rand_r = [coverage_radius(E, rng.choice(N, K, replace=False)) for _ in range(30)]
rand_b = [bucket_cov(rng.choice(N, K, replace=False)) for _ in range(30)]
rand_t = [tail_frac(rng.choice(N, K, replace=False)) for _ in range(30)]

print(f'{"选法":<16s} {"覆盖半径":>10s} {"覆盖场景组合":>14s} {"不同 clip":>10s} {"长尾占比":>10s}')
rows = [('core-set 贪心', coverage_radius(E, sel_cs), bucket_cov(sel_cs),
         len(set(clip_of[sel_cs].tolist())), tail_frac(sel_cs)),
        ('随机采样(均值)', float(np.mean(rand_r)), float(np.mean(rand_b)), None, float(np.mean(rand_t))),
        ('检索 top-K', coverage_radius(E, sel_topk), bucket_cov(sel_topk),
         len(set(clip_of[sel_topk].tolist())), tail_frac(sel_topk))]
for nm, rad, bc, nc, tf in rows:
    print(f'{nm:<16s} {rad:>10.3f} {bc:>14.1f} {str(nc) if nc else "-":>10s} {tf:>10.1%}')
print(f'（池子里长尾场景的自然占比 = {np.mean([(f["weather"], f["light"]) in tail for f in frames]):.1%}）')

assert coverage_radius(E, sel_cs) < 0.5 * np.mean(rand_r), 'core-set 的覆盖半径应显著更小'
assert bucket_cov(sel_cs) > bucket_cov(sel_topk), 'core-set 覆盖的场景组合应多于 top-K 检索'
assert tail_frac(sel_cs) > 2 * np.mean([(f['weather'], f['light']) in tail for f in frames])
print('\n✅ 同样 80 张预算：core-set 的覆盖半径只有随机的 1/5，长尾场景占比翻了两三倍，')
print('   而检索 top-K 只覆盖到极少数几个 clip —— **它找得准，但买得重复**。')
print('⚠️  k-center 的死穴：**对离群点极度敏感**。一帧相机故障的全绿图离所有点都远，')
print('   它一定会被选中，接着它旁边的噪声帧也会被选中 —— 预算被坏数据吃掉。')

In [ ]:
# 更稳健的替代：聚类/标签均衡采样（轮转配额，某层发完就跳过）
def balanced_sample(k, keyfn, seed=0):
    r = np.random.default_rng(seed)
    groups = collections.defaultdict(list)
    for i in range(N):
        groups[keyfn(frames[i])].append(i)
    for g in groups.values():
        r.shuffle(g)
    keys = sorted(groups, key=str)
    out, ptr = [], collections.defaultdict(int)
    while len(out) < k:
        moved = False
        for kk in keys:
            if ptr[kk] < len(groups[kk]) and len(out) < k:
                out.append(groups[kk][ptr[kk]]); ptr[kk] += 1; moved = True
        if not moved:
            break
    return np.array(out)

bal = balanced_sample(K, lambda f: (f['weather'], f['light']))
print(f'{"选法":<16s} {"覆盖场景组合":>14s} {"长尾占比":>10s} {"对离群点":>10s}')
print(f'{"core-set 贪心":<16s} {bucket_cov(sel_cs):>14d} {tail_frac(sel_cs):>10.1%} {"极敏感":>10s}')
print(f'{"标签均衡采样":<16s} {bucket_cov(bal):>14d} {tail_frac(bal):>10.1%} {"稳健":>10s}')
print(f'{"随机采样":<16s} {np.mean(rand_b):>14.1f} {np.mean(rand_t):>10.1%} {"稳健":>10s}')
assert bucket_cov(bal) >= bucket_cov(sel_cs) - 2
assert tail_frac(bal) > np.mean(rand_t)
print('\n✅ 均衡采样没有 k-center 激进，但对噪声稳健得多 —— **工业上更常见的选择**。')
print('   实战组合：先离群剔除 -> 再按场景标签分层 -> 层内用 core-set 或随机。')

## 6 · 场景打标：元数据 / 规则 / VLM 的三级级联

三条路各有覆盖面：**元数据免费但只能推部分轴**、**VLM 准但要花钱**。
级联的做法是：规则有把握的地方用规则（免费且往往更准），剩下的交给 VLM。

In [ ]:
# —— 车端可拿到的元数据（雨刷档位 / 大灯 / 时间 / 地图道路等级）——
def make_meta(f):
    pw = {'clear': [.93, .06, .01], 'rain': [.03, .22, .75],
          'fog':   [.25, .62, .13], 'snow': [.05, .25, .70]}[f['weather']]
    wiper = int(rng.choice(3, p=pw))
    l = f['light']
    if   l == 'day':   hour, hl = int(rng.integers(9, 17)), int(rng.random() < 0.08)
    elif l == 'dusk':  hour, hl = int(rng.integers(17, 20)), int(rng.random() < 0.65)
    elif l == 'night': hour, hl = int(rng.choice([20, 21, 22, 23, 0, 1, 2, 5])), 1
    else:              hour, hl = int(rng.integers(9, 17)), 1   # ← tunnel：白天却开灯（陷阱）
    return dict(wiper=wiper, headlight=hl, hour=hour, road_class=f['road'])

METAD = [make_meta(f) for f in frames]

def rule_tag(m):
    w = 'rain' if m['wiper'] >= 1 else 'clear'         # 规则**根本区分不出** fog / snow
    if   17 <= m['hour'] < 20:                  l = 'dusk'
    elif m['hour'] >= 20 or m['hour'] < 6:      l = 'night'
    elif m['headlight'] == 1:                   l = 'night'   # ← 隧道被误判成夜间
    else:                                       l = 'day'
    return w, l

def rule_confident(m):
    '''规则「有把握」的判据：只在元数据能唯一确定时才认。'''
    cw = (m['wiper'] == 0)                                        # 雨刷不动 -> 基本是 clear
    cl = (17 <= m['hour'] < 20) or m['hour'] >= 20 or m['hour'] < 6 or m['headlight'] == 0
    return cw, cl

# —— 模拟 VLM 打标：**错误不是随机的，稀有取值召回明显更低** ——
P_OK_W = {'clear': .96, 'rain': .88, 'fog': .62, 'snow': .75}
P_OK_L = {'day': .96, 'dusk': .80, 'night': .93, 'tunnel': .70}
CONF_W = {'clear': ['fog'], 'rain': ['fog', 'snow'], 'fog': ['rain', 'clear'], 'snow': ['rain']}
CONF_L = {'day': ['dusk'], 'dusk': ['day', 'night'], 'night': ['dusk', 'tunnel'], 'tunnel': ['night']}
VLM_COST = 0.0015                                    # 美元 / 帧

def vlm_tag(f):
    w = f['weather'] if rng.random() < P_OK_W[f['weather']] else str(rng.choice(CONF_W[f['weather']]))
    l = f['light']   if rng.random() < P_OK_L[f['light']]   else str(rng.choice(CONF_L[f['light']]))
    return w, l

rule_w, rule_l = zip(*[rule_tag(m) for m in METAD])
vlm_w,  vlm_l  = zip(*[vlm_tag(f) for f in frames])
gt_w = [f['weather'] for f in frames]; gt_l = [f['light'] for f in frames]
acc = lambda a, b: float(np.mean([x == y for x, y in zip(a, b)]))

cas_w, cas_l, ncall = [], [], 0
for i, f in enumerate(frames):
    cw, cl = rule_confident(METAD[i])
    rw, rl = rule_tag(METAD[i])
    if not (cw and cl):
        ncall += 1
    cas_w.append(rw if cw else vlm_w[i])
    cas_l.append(rl if cl else vlm_l[i])

print(f'{"方案":<14s} {"weather 准确率":>15s} {"lighting 准确率":>16s} {"VLM 调用率":>11s} {"成本(USD)":>11s}')
print(f'{"① 纯规则":<14s} {acc(rule_w, gt_w):>15.1%} {acc(rule_l, gt_l):>16.1%} {0.0:>11.1%} {0.0:>11.2f}')
print(f'{"③ 全量 VLM":<14s} {acc(vlm_w, gt_w):>15.1%} {acc(vlm_l, gt_l):>16.1%} {1.0:>11.1%} {N * VLM_COST:>11.2f}')
print(f'{"①+③ 级联":<14s} {acc(cas_w, gt_w):>15.1%} {acc(cas_l, gt_l):>16.1%} '
      f'{ncall / N:>11.1%} {ncall * VLM_COST:>11.2f}')
assert acc(vlm_w, gt_w) > acc(rule_w, gt_w), 'VLM 应显著强于纯规则'
assert ncall < 0.6 * N, '级联应大幅削减 VLM 调用量'
assert acc(cas_l, gt_l) > acc(vlm_l, gt_l), '元数据确定的部分比 VLM 更准'
print('\n✅ 级联不只是省钱：在元数据能唯一确定的那部分，规则**比 VLM 更准**（它是真值不是猜的）。')
print('   这就是「先免费的、再便宜的、最后才是贵的」这条工程铁律的具体形态。')

In [ ]:
# ⚠️ VLM 的错误是**成片的**：总体准确率好看，稀有取值的召回可能烂到不能用
def per_value_recall(pred, gt, labels):
    out = {}
    for L in labels:
        m = [i for i in range(len(gt)) if gt[i] == L]
        out[L] = (len(m), float(np.mean([pred[i] == L for i in m])) if m else float('nan'))
    return out

print('VLM 逐取值召回（weather）—— 总体准确率 %.1f%%：' % (100 * acc(vlm_w, gt_w)))
rec_w = per_value_recall(vlm_w, gt_w, WEATHER)
for L, (n_, r_) in rec_w.items():
    flag = '   ← ⚠️ 这一格的统计不能信' if r_ < 0.75 else ''
    print(f'  {L:<7s} n={n_:5d}  recall={r_:.2f}{flag}')
print('\nVLM 逐取值召回（lighting）—— 总体准确率 %.1f%%：' % (100 * acc(vlm_l, gt_l)))
rec_l = per_value_recall(vlm_l, gt_l, LIGHT)
for L, (n_, r_) in rec_l.items():
    flag = '   ← ⚠️' if r_ < 0.75 else ''
    print(f'  {L:<7s} n={n_:5d}  recall={r_:.2f}{flag}')

def cohen_kappa(a, b, labels):
    '''一致性系数：扣掉「碰巧同意」之后的一致程度。'''
    po = float(np.mean([x == y for x, y in zip(a, b)]))
    pe = float(sum(np.mean([x == L for x in a]) * np.mean([y == L for y in b]) for L in labels))
    return (po - pe) / (1 - pe)

audit = rng.choice(N, 300, replace=False)              # 送人工的**审计集**
k_vlm  = cohen_kappa([vlm_w[i] for i in audit],  [gt_w[i] for i in audit], WEATHER)
k_rule = cohen_kappa([rule_w[i] for i in audit], [gt_w[i] for i in audit], WEATHER)
print(f'\n审计集 300 帧的 Cohen kappa（weather）：VLM {k_vlm:.3f}   纯规则 {k_rule:.3f}')
assert rec_w['fog'][1] < 0.80, 'fog 是稀有取值，VLM 召回应明显偏低'
assert k_vlm > k_rule and k_vlm > 0.6
print('\n⚠️  总体准确率 90% 看着很好，但 fog 只有 ~60% 的召回 ——')
print('    于是「我们池子里雾天数据不多」这个结论**是标注器造出来的假象**。')
print('✅ 所以审计集不能只看总体准确率，必须**按每个取值分别算召回**，专盯稀有取值。')
print('   经验门槛：kappa < 0.6 的标签轴，不要拿它做任何数据决策。')

## 7 · 标注预算分配：把钱花在最缺的格子上

把「标 n 张后该格的收益」建模成饱和曲线 $V_c(n)=w_c(1-e^{-n/\tau_c})$，
则边际收益率 = $\dfrac{w_c}{\tau_c\,\text{cost}_c}e^{-n/\tau_c}$。
因为 $V_c$ 是凹的，「每次把下一块钱给边际收益率最高的格子」这个贪心是最优的。

In [ ]:
CELLS = [
    #  格子名                     已有   重要度  饱和尺度  单价(元)  池中可挖
    dict(name='tunnel × construction', have=180,   w=9.0,  tau=1500, cost=3.2, avail=4000),
    dict(name='fog × speed_limit',     have=90,    w=6.0,  tau=1200, cost=2.8, avail=1500),
    dict(name='day_urban × speed',     have=42000, w=10.0, tau=8000, cost=1.0, avail=200000),
    dict(name='rain × no_left',        have=600,   w=4.0,  tau=2000, cost=2.0, avail=9000),
    dict(name='night × stop',          have=1500,  w=8.0,  tau=3000, cost=1.8, avail=12000),
    dict(name='snow × any',            have=25,    w=2.5,  tau=800,  cost=4.0, avail=300),
]
CHUNK, BUDGET, MIN_RATE = 50, 30000.0, 1e-5

def total_value(alloc):
    return sum(c['w'] * (1 - math.exp(-(c['have'] + alloc[c['name']]) / c['tau'])) for c in CELLS)
BASE_V = sum(c['w'] * (1 - math.exp(-c['have'] / c['tau'])) for c in CELLS)

def alloc_greedy(budget, min_rate=MIN_RATE):
    x = {c['name']: 0 for c in CELLS}; spent = 0.0
    while True:
        best, rate = None, min_rate            # ← 低于门槛就不投：**预算没花完是对的**
        for c in CELLS:
            if x[c['name']] + CHUNK > c['avail']:      continue
            price = c['cost'] * CHUNK
            if spent + price > budget:                 continue
            n0 = c['have'] + x[c['name']]
            dv = c['w'] * (math.exp(-n0 / c['tau']) - math.exp(-(n0 + CHUNK) / c['tau']))
            if dv / price > rate:
                best, rate = c, dv / price
        if best is None:
            break
        x[best['name']] += CHUNK; spent += best['cost'] * CHUNK
    return x, spent

def alloc_uniform(budget):
    x = {c['name']: int(min(c['avail'], (budget / len(CELLS)) // c['cost'])) for c in CELLS}
    return x, sum(x[c['name']] * c['cost'] for c in CELLS)

def alloc_lowest_first(budget):
    '''常见但错误的做法：哪个格子最缺就全给它。'''
    x = {c['name']: 0 for c in CELLS}; spent = 0.0
    for c in sorted(CELLS, key=lambda c: c['have'] / c['tau']):
        n = int(min(c['avail'], (budget - spent) // c['cost']))
        x[c['name']] = n; spent += n * c['cost']
    return x, spent

print(f'{"分配策略":<18s} {"总收益":>10s} {"花费(元)":>11s}')
allocs = {}
for nm, fn in [('贪心：边际/元', alloc_greedy), ('平均分', alloc_uniform), ('缺口最大优先', alloc_lowest_first)]:
    x, sp = fn(BUDGET); allocs[nm] = x
    print(f'{nm:<18s} {total_value(x) - BASE_V:>10.3f} {sp:>11.0f}')
print(f'\n贪心的具体分配：')
for c in CELLS:
    n = allocs['贪心：边际/元'][c['name']]
    note = '   ← **已饱和，一分钱不给**' if n == 0 else ('   ← 池子里就这么多' if n >= c['avail'] else '')
    print(f'  {c["name"]:<24s} {n:>6d} 张  花 {n * c["cost"]:>7.0f} 元{note}')

g = total_value(allocs['贪心：边际/元'])
assert g > total_value(allocs['平均分']), '贪心应优于平均分'
assert g > total_value(allocs['缺口最大优先']), '贪心应优于「哪个最缺给哪个」'
assert allocs['贪心：边际/元']['day_urban × speed'] == 0, '已饱和的格子不该拿到预算'
print('\n✅ 三个符合直觉的行为自动出现了：已饱和的格子拿 0；重要度高的多给；单价贵的被折价。')
print('⚠️  这个模型的两个漏洞要能说出来：① 新场景没有历史 -> 没有 tau，只能先给探索性小配额；')
print('    ② 格子之间不独立（夜间限速牌涨了，夜间禁令牌也会涨）-> 贪心会高估分散投资的价值。')
print('⚠️  还要留 10~15% 预算，按**部署真实分布**随机采样去标，维护一个不漂移的基准评测集。')

## ✏️ 练习 1：MMR —— 相关性与多样性的显式权衡

实现 `mmr_select(sims_q, X, cand, k, lam)`（Maximal Marginal Relevance）：

- 第一个**总是**选 `sims_q` 最大的候选；
- 之后每步选使 `lam * sims_q[i] - (1 - lam) * max_{j∈S} X[i]·X[j]` 最大的 `i`；
- 已选过的不再选；返回长度为 `k` 的 numpy 整数数组（按选中顺序）。

`lam=1.0` 时应退化成「按相关性取 top-k」。

In [ ]:
def mmr_select(sims_q, X, cand, k, lam):
    # TODO: ① 第一个选 sims_q 最大的
    #       ② 之后每步最大化 lam*相关性 - (1-lam)*与已选集合的最大相似度
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
q_sims = E @ E[qi]
cand = np.argsort(-q_sims)[:200]
sel_lam1 = mmr_select(q_sims, E, cand, 8, 1.0)
assert list(map(int, sel_lam1)) == list(map(int, cand[:8])), 'lam=1 应退化成 top-k'
sel_mmr = mmr_select(q_sims, E, cand, 8, 0.35)
assert len(set(map(int, sel_mmr))) == 8, '不能重复选'
n1 = len({int(clip_of[i]) for i in sel_lam1})
n2 = len({int(clip_of[i]) for i in sel_mmr})
print(f'lam=1.00（纯相关性）选出的 8 张来自 {n1} 个 clip')
print(f'lam=0.35（相关+多样）选出的 8 张来自 {n2} 个 clip')
assert n2 > n1, 'MMR 应该显著提高来源多样性'
print('✅ 练习 1 通过：**一个 lam 就把「找得准」和「买得散」放到了同一个目标函数里**')

## ✏️ 练习 2：分层配额（stratified quota）

实现 `stratified_quota(keys, k)`：`keys` 是每个候选样本所属的层标签列表，`k` 是总配额。

- 各层配额尽量均等；
- **某层样本数不够时，余额要分给还有余量的层**（不能浪费配额）；
- 返回 `{层: 配额}`，且 `sum(配额) == min(k, len(keys))`；配额不超过该层样本数。

提示：按「该层样本数」升序轮转发放，每轮每层发 1 张，发满就跳过。

In [ ]:
def stratified_quota(keys, k):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
keys = ['A'] * 100 + ['B'] * 100 + ['C'] * 3 + ['D'] * 1
q = stratified_quota(keys, 40)
assert sum(q.values()) == 40, q
assert q['C'] == 3 and q['D'] == 1, '样本不够的层应全拿，且不超发'
assert abs(q['A'] - q['B']) <= 1 and q['A'] + q['B'] == 36, q
q2 = stratified_quota(keys, 10000)
assert sum(q2.values()) == len(keys) and q2['A'] == 100
q3 = stratified_quota(keys, 0)
assert sum(q3.values()) == 0
real = stratified_quota([(f['weather'], f['light']) for f in frames], 80)
print('真实池子上按 (天气,光照) 分层的 80 张配额（非零项）:')
for kk, v in sorted(real.items(), key=lambda kv: -kv[1]):
    if v:
        print(f'  {str(kk):<22s} {v:3d}')
assert sum(real.values()) == 80 and len([v for v in real.values() if v]) >= 8
print('✅ 练习 2 通过：**分层配额是最容易落地、也最难出错的多样性工具**')

## ✏️ 练习 3：近重复分组（连通分量版去重）

贪心去重是**顺序相关**的（先来后到决定谁当代表）。更稳的做法是把
「汉明距离 ≤ thresh」看成一张图的边，取**连通分量**作为重复组。

实现 `near_dup_groups(H, thresh)`：返回 `list[list[int]]`，每个内层列表是一组的下标（升序），
外层按每组最小下标升序。要求：每个样本恰好属于一组，组的并集是全体。

提示：并查集（union-find）；先算两两汉明距离矩阵（本例 N 只有两千量级，可以直接算）。

In [ ]:
def near_dup_groups(H, thresh):
    # TODO: 并查集 + 汉明距离 <= thresh 连边 -> 连通分量
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
groups = near_dup_groups(H, 3)
flat = [i for g in groups for i in g]
assert sorted(flat) == list(range(len(H))), '每个样本恰好属于一组'
assert all(list(g) == sorted(g) for g in groups) and groups == sorted(groups, key=lambda g: g[0])
assert len(groups) < len(H) // 3, f'去重后组数应远小于总帧数，实际 {len(groups)}'
gid = {i: k for k, g in enumerate(groups) for i in g}
pair = next((a, b) for idxs in by_clip.values() for a, b in itertools.combinations(idxs, 2)
            if np.count_nonzero(H[a] != H[b]) <= 3)
assert gid[pair[0]] == gid[pair[1]], '汉明距离 <= 3 的两帧必须在同一组'

print(f'{"thresh":>7s} {"组数":>7s} {"最大组":>8s} {"单例组":>8s}')
prev = None
for T in [1, 2, 3, 4, 5, 6]:
    gs = near_dup_groups(H, T)
    sz = sorted((len(x) for x in gs), reverse=True)
    flag = '   ← ⚠️ **链式合并塌方**' if sz[0] > len(H) // 4 else ''
    print(f'{T:>7d} {len(gs):>7d} {sz[0]:>8d} {sum(1 for s in sz if s == 1):>8d}{flag}')
    if T == 6:
        prev = sz[0]
assert prev > len(H) // 4, '阈值放大后应出现巨型连通分量'
print(f'\n贪心去重(thresh=6) 保留 {len(kept6)} 帧 vs 连通分量(thresh=3) {len(groups)} 组')
print('✅ 练习 3 通过：**连通分量与顺序无关**，但会因为「链式合并」比贪心更激进 ——')
print('   A~B、B~C 但 A 与 C 完全不像时，三者仍会被并成一组。阈值从 3 放到 6，')
print('   最大组就从几十帧膨胀到整个池子的一半 —— **所以连通分量法的阈值必须更保守**。')

## ✏️ 练习 4：数据血缘 —— 「这批数据出问题了，谁受影响？」

给定数据集版本表与训练任务表，实现两个函数：

- `batches_of(datasets, ds)`：返回该数据集版本**递归展开父版本后**包含的全部 batch（集合）；
- `affected_models(datasets, runs, bad_batch)`：返回用到了 `bad_batch` 的**全部** model id（升序列表）。

这是数据闭环出事故时第一个要能回答的问题——没有它，只能全量重训「以防万一」。

In [ ]:
DATASETS = {
    'v1.7.1': dict(parent=None,     batches=['b_base_2025q4', 'b_night_2026w12']),
    'v1.7.2': dict(parent='v1.7.1', batches=['b_tunnel_2026w27']),
    'v1.7.3': dict(parent='v1.7.2', batches=['b_fog_2026w29']),
    'v2.0.0': dict(parent=None,     batches=['b_base_2025q4', 'b_resample_2026w30']),
}
RUNS = {'model_r31': 'v1.7.1', 'model_r38': 'v1.7.2',
        'model_r41': 'v1.7.3', 'model_x02': 'v2.0.0'}

def batches_of(datasets, ds):
    # TODO
    raise NotImplementedError

def affected_models(datasets, runs, bad_batch):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert batches_of(DATASETS, 'v1.7.1') == {'b_base_2025q4', 'b_night_2026w12'}
assert batches_of(DATASETS, 'v1.7.3') == {'b_base_2025q4', 'b_night_2026w12',
                                          'b_tunnel_2026w27', 'b_fog_2026w29'}
assert affected_models(DATASETS, RUNS, 'b_tunnel_2026w27') == ['model_r38', 'model_r41']
assert affected_models(DATASETS, RUNS, 'b_fog_2026w29') == ['model_r41']
assert affected_models(DATASETS, RUNS, 'b_base_2025q4') == \
       ['model_r31', 'model_r38', 'model_r41', 'model_x02']
assert affected_models(DATASETS, RUNS, 'b_not_exist') == []
for b in ['b_tunnel_2026w27', 'b_base_2025q4']:
    print(f'{b:<22s} 受影响模型: {affected_models(DATASETS, RUNS, b)}')
print('✅ 练习 4 通过：**递归展开父版本**是最容易漏掉的一步 ——')
print('   只查「直接引用」会漏掉所有继承自它的下游数据集与模型。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def mmr_select(sims_q, X, cand, k, lam):
    cand = np.asarray(cand)
    first = int(cand[int(np.argmax(sims_q[cand]))])
    sel = [first]
    remaining = [int(i) for i in cand if int(i) != first]
    max_sim = {i: float(X[i] @ X[first]) for i in remaining}   # 到已选集合的最大相似度
    while len(sel) < k and remaining:
        scores = [lam * float(sims_q[i]) - (1 - lam) * max_sim[i] for i in remaining]
        pick = remaining.pop(int(np.argmax(scores)))
        sel.append(pick)
        for i in remaining:
            max_sim[i] = max(max_sim[i], float(X[i] @ X[pick]))
    return np.array(sel, dtype=int)

In [ ]:
# 练习 2 参考答案
def stratified_quota(keys, k):
    caps = collections.Counter(keys)
    quota = {g: 0 for g in caps}
    order = sorted(caps, key=lambda g: (caps[g], str(g)))       # 小层优先，保证余额能流出去
    left = min(k, len(keys))
    while left > 0:
        moved = False
        for g in order:
            if left == 0:
                break
            if quota[g] < caps[g]:
                quota[g] += 1; left -= 1; moved = True
        if not moved:
            break
    return quota

In [ ]:
# 练习 3 参考答案
def near_dup_groups(H, thresh):
    n = len(H)
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    Hi = H.astype(np.int8)
    for i in range(n):                                   # 一行对全体，向量化算汉明距离
        d = np.count_nonzero(Hi[i + 1:] != Hi[i], axis=1)
        for off in np.where(d <= thresh)[0]:
            union(i, i + 1 + int(off))
    buckets = collections.defaultdict(list)
    for i in range(n):
        buckets[find(i)].append(i)
    return sorted((sorted(v) for v in buckets.values()), key=lambda g: g[0])

In [ ]:
# 练习 4 参考答案
def batches_of(datasets, ds):
    out, cur = set(), ds
    while cur is not None:
        out |= set(datasets[cur]['batches'])
        cur = datasets[cur]['parent']
    return out

def affected_models(datasets, runs, bad_batch):
    return sorted(m for m, ds in runs.items() if bad_batch in batches_of(datasets, ds))

---
## 🧪 真实工程胶囊：一条可直接照搬的挖掘任务定义

In [ ]:
RECIPE = r'''
# ===================== mining_job.yaml =====================
job_id: mine_2026w31_tunnel_construction
owner: perception-data@
# ---- ① 触发来源：这次挖掘为了修什么 ----
motivation:
  badcase_ticket: TSR-4471            # 路测单号，必填。没有 ticket 的挖掘任务不予排期
  failure_mode:   "隧道出口逆光下施工牌漏检（首检距离 < 25m）"
  seed_frames:    [frame_88e1a, frame_9d02c, frame_31fb7]

# ---- ② 召回：结构化过滤 **先** 于向量检索（先砍掉 99%）----
filter:
  tags.lighting: [tunnel, night]
  tags.sign_class: [construction, temporary]
  tags.tag_version: ">=2026.06"       # 标签口径版本，防止跨口径统计
retrieve:
  index:        ivf_pq_v3             # **索引绑定嵌入模型版本**
  embedding:    det_backbone@f19c2    # 换 backbone 必须重建索引，否则静默返回垃圾
  metric:       cosine                # 建索引前已 L2 归一化
  nprobe:       16                    # 召回 0.98 @ 12% 算力（离线批量任务可以调大）
  top_k:        20000

# ---- ③ 精选：去重 -> 多样性 -> 配额 ----
dedup:
  phash:        {type: phash64, thresh: 6}      # 先砍像素级近重复
  embed_radius: {cosine: 0.06, within_tag_only: true}  # **跨场景不去重**
diversify:
  method:       coreset_kcenter
  outlier_filter: {knn_density_percentile: 2}   # k-center 对离群点极敏感 -> 先剔除
  k:            1200
quota:
  stratify_by:  [tags.weather, tags.lighting]
  reserve_random_frac: 0.12           # **留 12% 按真实分布随机采**，防评测集漂移

# ---- ④ 送标 ----
label:
  spec_version: tsr_label_spec_v4.2
  price_per_frame_cny: {default: 2.0, occlusion_heavy: 3.2, night: 2.6}
  budget_cny:   30000
  qc_sample_rate: 0.05                # 质检抽样率；通过率 < 0.92 整批打回

# ---- ⑤ 产物与血缘（这一段不是文档，是**机器可读的契约**）----
outputs:
  batch_id:     b_tunnel_2026w31
  lineage:
    mined_by:   {job: mine_2026w31_tunnel_construction, index: ivf_pq_v3,
                 embedding: det_backbone@f19c2, seeds: [frame_88e1a, ...], rng_seed: 7}
    split_key:  clip_id               # **按 clip 划分训练/评测，绝不按帧**（防泄漏）
gate:
  - qc_pass_rate >= 0.92
  - duplicate_rate <= 0.03            # 抽检重复率
  - tag_kappa(weather) >= 0.6         # 标签一致性不达标 -> 这批标签不参与统计

# ===================== VLM 打标 prompt =====================
# 关键三条：JSON schema 强约束 / 保留 unknown 与 confidence / 逐取值召回做审计
VLM_PROMPT = (
  "只输出 JSON。字段与允许取值：\n"
  '  weather:    ["clear","rain","fog","snow","unknown"]\n'
  '  lighting:   ["day","dusk","night","tunnel","unknown"]\n'
  '  occlusion:  ["none","partial","heavy","unknown"]\n'
  "  confidence: 0.0-1.0\n"
  "无法判断填 unknown，不要猜。"          # 强迫二选一会制造大量静默错误
)
'''
print(RECIPE)
for token in ['badcase_ticket', 'nprobe', 'phash', 'coreset_kcenter',
              'reserve_random_frac', 'split_key', 'tag_kappa', 'unknown']:
    assert token in RECIPE, token
print('✅ 配方覆盖：ticket 溯源 / 标签先过滤 / 索引绑嵌入版本 / 去重 / core-set / '
      '分层配额 / 随机保留集 / 质检门禁 / clip 级划分 / 血缘')

### 小结

- **挖掘系统的产出不是「找到多少难例」，而是「每块标注预算换来多少模型提升」**。
  候选池免费、标注昂贵，所以全部设计压力都在「筛选」这一步。
- **相似度决定「找什么」，多样性决定「买什么」**。把这两阶段目标搞混 = 回传 1000 张同一路口。
  实测：同样 80 张预算，core-set 的覆盖半径只有随机的 1/5，而检索 top-K 只覆盖到几个 clip。
- **嵌入检索前必须 L2 归一化**（否则排序被候选的模长劫持）；**索引必须绑定嵌入模型版本**
  ——换 backbone 后旧索引不在同一空间，检索会静默返回垃圾。
- **场景标签体系 = 正交轴 + 受控词表 + 来源与置信度**。它的最大价值是能算出**缺口矩阵**，
  把「模型不太行」变成「`(tunnel) × construction` 只有 180 张，AP 0.34」。
- **VLM 打标是当前主流**，因为它把「加一个新标签轴」从「训个分类器」降成「改段 prompt」。
  但它的错误是**成片的**：总体 90% 而 fog 召回只有 60%——审计必须按取值分别算召回，kappa &lt; 0.6 不用。
- **三级级联**（元数据 → 规则/小模型 → VLM）能把 VLM 调用量降到 ~40%，
  而且在元数据能唯一确定的部分**比 VLM 更准**。
- **预算分配 = 带成本的凹函数贪心**：已饱和的格子给 0、重要度高的多给、单价贵的折价；
  并且**预算没花完是对的**，边际收益率低于门槛就该留到下一轮。
- **血缘的三张表 + 一条纪律**：batch / dataset / run，训练脚本只接受数据集版本号。
  **划分训练-评测必须按 clip 而不是按帧**——TSR 里一块标志会连续出现在几十帧，按帧划分会让指标虚高。

下一站：**模块 05 · 闭环验证** —— 数据挖回来标完了，怎么证明它<em>真的</em>有用。